In [1]:
import pandas as pd
import numpy as np
from ehull_utils import (
    collect_mlip_energies_to_df,
    construct_phase_diagrams,
    summarize_results,
)
import os
import glob
from pymatgen.core import Structure

In [2]:
hdp_df = pd.read_csv(
    "/home/lwalterb/hdp_project/HDP_WorkFlow_Analysis/E_above_hull/HDP_CombinedInfo_WithChemSys.csv",
    index_col=0,
)
hdp_df.set_index("comp", inplace=True)
subsys_df = pd.read_csv(
    "/home/lwalterb/hdp_project/HDP_WorkFlow_Analysis/E_above_hull/hdp_mp_subsysandhdps.csv",
    index_col=0,
)
subsys_df["structure"] = subsys_df.apply(
    lambda row: Structure.from_dict(eval(row["structure_dict"])), axis=1
)
# mlip_df = pd.read_csv("/home/lwalterb/hdp_project/HDP_WorkFlow_Analysis/E_above_hull/hdp_mlipenergies_full.csv",index_col=0)
# mlip_df['structure'] = mlip_df.apply(lambda row: Structure.from_dict(eval(row['structure_dict'])),axis=1)

In [3]:
data_dirs = ["ExpensiveMLIPs"]
for data_dir in data_dirs:
    # mlip_list = [x.split("_")[-1].strip('.json') for x in glob.glob(f'{data_dir}/*.json')]
    mlip_list = [
        "GRACE-3L-OMAT-large-ft-AM",
        "SevenNet-MPALOE",
        "SevenNet-omat24",
        "PET-OAM-XL",
    ]  # ,   ['Equiformer-v3']
    print(mlip_list)
    print(f"collecting data from {data_dir}")
    mlip_df = collect_mlip_energies_to_df(
        structures_df=subsys_df,
        mlip_list=mlip_list,
        data_dir=data_dir,
        output_fn=os.path.join(data_dir, f"hdp_mlipenergies_{len(mlip_list)}MLIPS.csv"),
    )
    # Failed convergence strings and NaNs need to be replaces with None to assure functionality
    mask = mlip_df.astype(str).apply(lambda col: col.str.contains("Failed", na=True))
    mlip_df = mlip_df.mask(mask, None)
    # print(mlip_df.head())
    print(f"mlipdata_shape: {mlip_df.shape}")
    print(f"Collecting complete, starting phase diagram construction...")
    ehull_df, eform_df = construct_phase_diagrams(
        hdp_df=hdp_df,
        subsys_MLIPenergy_df=mlip_df,
        dataframe_savedir=data_dir,
        phasediagram_savedir= None #os.path.join(data_dir, "PhaseDiagramData"),
    )
    print(f"Ehull calculation complete, simplified report from {data_dir}:")

    summarize_results(mlip_df, ehull_df)

['GRACE-3L-OMAT-large-ft-AM', 'SevenNet-MPALOE', 'SevenNet-omat24', 'PET-OAM-XL']
collecting data from ExpensiveMLIPs
mlipdata_shape: (12154, 8)
Ehull calculation complete, simplified report from ExpensiveMLIPs:
MLIP: 	 NaN vals:
E_GRACE-3L-OMAT-large-ft-AM 	 258
E_SevenNet-MPALOE 	 3
E_SevenNet-omat24 	 2
E_PET-OAM-XL 	 2
MLIP: 	 Missing HDPs: 	 share stable (<=100meV/atom):
GRACE-3L-OMAT-large-ft-AM 	 258 	 0.7237302977232924
SevenNet-MPALOE 	 259 	 0.7459483136224266
SevenNet-omat24 	 258 	 0.6957092819614711
PET-OAM-XL 	 258 	 0.7092819614711033
MLIP: 	 Missing HDPs: 	 share stable (<=150meV/atom):
GRACE-3L-OMAT-large-ft-AM 	 258 	 0.8489492119089317
SevenNet-MPALOE 	 259 	 0.8576434515987735
SevenNet-omat24 	 258 	 0.8239929947460596
PET-OAM-XL 	 258 	 0.8226795096322241
MLIP: 	 Missing HDPs: 	 share stable (<=200meV/atom):
GRACE-3L-OMAT-large-ft-AM 	 258 	 0.9001751313485113
SevenNet-MPALOE 	 259 	 0.9093298291721419
SevenNet-omat24 	 258 	 0.8905429071803853
PET-OAM-XL 	 258 	 0

In [ ]:
for col in mlip_df.columns:
    if col.startswith("E_"):
        miss_data = mlip_df.loc[mlip_df[col].isna()]
        miss_data[["chemsys", "nsites", "structure_dict"]].to_csv(
            f"MissingStrucs_{col.removeprefix('E_')}.csv"
        )

In [ ]:
allowed_els = [
    "Ac",
    "Ag",
    "Al",
    "Ar",
    "As",
    "Au",
    "B",
    "Ba",
    "Be",
    "Bi",
    "Br",
    "C",
    "Ca",
    "Cd",
    "Ce",
    "Cl",
    "Co",
    "Cr",
    "Cs",
    "Cu",
    "Dy",
    "Er",
    "Eu",
    "F",
    "Fe",
    "Ga",
    "Gd",
    "Ge",
    "H",
    "He",
    "Hf",
    "Hg",
    "Ho",
    "I",
    "In",
    "Ir",
    "K",
    "Kr",
    "La",
    "Li",
    "Lu",
    "Mg",
    "Mn",
    "Mo",
    "N",
    "Na",
    "Nb",
    "Nd",
    "Ne",
    "Ni",
    "Np",
    "O",
    "Os",
    "P",
    "Pa",
    "Pb",
    "Pd",
    "Pm",
    "Pr",
    "Pt",
    "Pu",
    "Rb",
    "Re",
    "Rh",
    "Ru",
    "S",
    "Sb",
    "Sc",
    "Se",
    "Si",
    "Sm",
    "Sn",
    "Sr",
    "Ta",
    "Tb",
    "Tc",
    "Te",
    "Th",
    "Ti",
    "Tl",
    "Tm",
    "U",
    "V",
    "W",
    "Xe",
    "Y",
    "Yb",
    "Zn",
    "Zr",
]
drop_index = mlip_df.apply(
    lambda row: False in [x in allowed_els for x in row["chemsys"].split("-")], axis=1
)
mlip_df.drop(drop_index[drop_index == True].index)

,structure_dict,chemsys,nsites,structure,E_SevenNet-MPALOE,E_SevenNet-omat24,E_PET-OAM-XL
mp-1225895,"{'@module': 'pymatgen.core.structure', '@class...",Cs-Rb,2,"[[0. 0. 0.] Cs, [3.03786378 0. 4.19497...",-39.01326,-1.790802,-1.804725
mp-1185559,"{'@module': 'pymatgen.core.structure', '@class...",Cs-Rb,4,"[[0. 0. 0.] Cs, [2.92420802 2.92420802 2.92420...",-66.810379,-3.631001,-3.744776
mp-1183945,"{'@module': 'pymatgen.core.structure', '@class...",Cs-Rb,4,"[[0. 0. 0.] Cs, [0. 3.69598298 3.69598...",-66.85466,-3.674257,-3.716936
mp-1184016,"{'@module': 'pymatgen.core.structure', '@class...",Cs-Rb,4,"[[0. 0. 0.] Cs, [4.44089210e-16 3.70126506e+00...",-66.857254,-3.677019,-3.744331
mp-862689,"{'@module': 'pymatgen.core.structure', '@class...",Cs-Rb,8,[[-6.10753338e-06 3.19002770e+00 2.22351732e...,-178.404694,-6.982025,-7.048181
...,...,...,...,...,...,...,...
3749_CsCsIrBr,"{'@module': 'pymatgen.core.structure', '@class...",Br-Cs-Ir,10,"[[0. 0. 0.] Cs, [5.73456991 5.73456991 5.73456...",-220.834442,-33.010117,-32.988205
3750_CsCsAuF,"{'@module': 'pymatgen.core.structure', '@class...",Au-Cs-F,10,"[[0. 0. 0.] Cs, [4.83390697 4.83390697 4.83390...",-166.809906,-37.208961,-37.719357
3751_CsCsAuCl,"{'@module': 'pymatgen.core.structure', '@class...",Au-Cl-Cs,10,"[[0. 0. 0.] Cs, [5.5867253 5.5867253 5.5867253...",-176.654877,-29.173389,-29.201574
3752_CsCsTlF,"{'@module': 'pymatgen.core.structure', '@class...",Cs-F-Tl,10,"[[0. 0. 0.] Cs, [4.90913393 4.90913393 4.90913...",-173.210083,-39.885536,-40.546158


In [4]:
ehull_df = pd.read_csv(
    "/home/lwalterb/hdp_project/HDP_WorkFlow_Analysis/E_above_hull/ExpensiveMLIPs/Ehull_data_GRACE-3L-OMAT-large-ft-AM_SevenNet-MPALOE_SevenNet-omat24_PET-OAM-XL.csv",
    index_col=0,
)
eform_df = pd.read_csv(
    "/home/lwalterb/hdp_project/HDP_WorkFlow_Analysis/E_above_hull/ExpensiveMLIPs/Eform_data_GRACE-3L-OMAT-large-ft-AM_SevenNet-MPALOE_SevenNet-omat24_PET-OAM-XL.csv",
    index_col=0,
)

ehull_avg = ehull_df.mean(axis=1)
ehull_avg.name = "Ehull_avg"
eform_avg = eform_df.mean(axis=1)
eform_avg.name = "Eform_avg"

ehull_std = ehull_df.std(axis=1)
ehull_std.name = "Ehull_std"
eform_std = eform_df.std(axis=1)
eform_std.name = "Eform_std"

In [8]:
ehull_df.dropna(how='all').shape

(2286, 4)

In [ ]:
def stable_counts(
    ehull_df: pd.DataFrame, cutoff_values: list[float] = [100, 150, 200]
) -> pd.DataFrame:
    counts_df = pd.DataFrame(index=ehull_df.index)
    counts_df["MLIP_entries"] = ehull_df.count(axis=1)
    for cutoff in cutoff_values:
        val = cutoff / 1000
        counts_df[f"Ehull <= {cutoff}meV"] = ehull_df.where(ehull_df < val).count(
            axis=1
        )

    return counts_df

In [ ]:
counts = stable_counts(ehull_df)
stab_data = pd.concat([eform_avg, eform_std, ehull_avg, ehull_std, counts], axis=1)
stab_data.to_csv("HDP_Ehull_overview.csv")

In [14]:
hdp_df.columns.to_list()

['Unnamed: 0',
 'compID_num',
 'comp_name_simple',
 'comp_name_full',
 'specie.A',
 'specie.B1',
 'specie.B2',
 'specie.X',
 'element.A',
 'element.B1',
 'element.B2',
 'element.X',
 'r_ionic.A',
 'r_ionic.B1',
 'r_ionic.B2',
 'r_ionic.X',
 'oct_factor',
 'oct_mismatch',
 'gen_t_factor',
 'geom_stable',
 'broken_condition',
 'tau_factor',
 'block.B1',
 'row.B1',
 'inputcharge.B1',
 'block.B2',
 'row.B2',
 'inputcharge.B2',
 'used_lobbasis',
 'used_lobbasis_func',
 'block_pairing',
 'VBM',
 'CBM',
 'bandgap',
 'VBMtotcontr.A',
 'VBMorbital.A',
 'VBMorbchar.A',
 'VBMorborder.A',
 'VBMorbcontr.A',
 'CBMtotcontr.A',
 'CBMorbital.A',
 'CBMorbchar.A',
 'CBMorborder.A',
 'CBMorbcontr.A',
 'VBMtotcontr.B1',
 'VBMorbital.B1',
 'VBMorbchar.B1',
 'VBMorborder.B1',
 'VBMorbcontr.B1',
 'CBMtotcontr.B1',
 'CBMorbital.B1',
 'CBMorbchar.B1',
 'CBMorborder.B1',
 'CBMorbcontr.B1',
 'VBMtotcontr.B2',
 'VBMorbital.B2',
 'VBMorbchar.B2',
 'VBMorborder.B2',
 'VBMorbcontr.B2',
 'CBMtotcontr.B2',
 'CBMorbital

In [ ]:
mlip_list = [
    "GRACE-3L-OMAT-large-ft-AM",
    "SevenNet-MPALOE",
    "SevenNet-omat24",
    "PET-OAM-XL",
]  # ,   ['Equiformer-v3']
missing_list = []
comps = {}
for mlip in mlip_list:
    misidx = pd.read_csv(f"MissingStrucs_{mlip}.csv", index_col=0)
    miss_ser = pd.Series(
        data={idx: np.int64(1) for idx in misidx.index if "_Cs" not in idx}, name=mlip
    )
    comps.update(
        {
            idx: Structure.from_dict(
                eval(misidx.loc[idx]["structure_dict"])
            ).composition.reduced_formula
            for idx in misidx.index
            if "_Cs" not in idx
        }
    )
    missing_list.append(miss_ser)
comp_series = pd.Series(comps, name="composition")
missing_list.append(comp_series)
missing_df = pd.DataFrame(missing_list).fillna(np.int64(0)).T
missing_df.to_csv("HDP_Ehull_MissingStrucs.csv")
missing_df

,GRACE-3L-OMAT-large-ft-AM,SevenNet-MPALOE,SevenNet-omat24,PET-OAM-XL,composition
mp-1214068,1.0,1.0,0,0,Cr3CdF6
mp-1212233,1.0,0,1.0,0,Mn3PtF6
mp-1207330,0,1.0,0,1.0,Hg3PtF6
mp-1212235,0,1.0,0,0,PdPt3F6
mp-1211674,0,0,1.0,0,Li3PF6
mp-1205972,0,0,0,1.0,MnZn3F6


In [ ]:
plot_df = pd.concat(
    [
        hdp_df[
            [
                "element.B1",
                "element.B2",
                "element.X",
                "tau_factor",
                "gen_t_factor",
                "oct_mismatch",
            ]
        ],
        ehull_df,
        ehull_avg,
        eform_avg,
    ],
    axis=1,
)
plot_df.head()

,element.B1,element.B2,element.X,tau_factor,gen_t_factor,oct_mismatch,GRACE-3L-OMAT-large-ft-AM,SevenNet-MPALOE,SevenNet-omat24,PET-OAM-XL,Ehull_avg,Eform_avg
1000_CsAgLuCl,Ag,Lu,Cl,3.787884,0.926126,0.079834,0.000000,0.000000,0.000000,0.000000,0.000000,-1.918685
1001_CsAuVCl,Au,V,Cl,3.787892,0.923029,0.201657,0.149100,0.129796,0.160484,0.552058,0.247860,-1.335096
1002_CsNaPrCl,Na,Pr,Cl,3.787892,0.926894,0.008287,0.000658,0.003897,0.000000,0.000113,0.001167,-2.150775
1003_CsAgDyCl,Ag,Dy,Cl,3.787892,0.926777,0.035912,0.000000,0.000000,0.000000,0.000000,0.000000,-1.917249
1004_CsMgNdCl,Mg,Nd,Cl,3.787892,0.924534,0.157459,0.127597,0.088584,0.139095,0.129547,0.121206,-1.969830


In [ ]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "browser"

# fig = px.scatter(plot_df, 'tau_factor','gen_t_factor',color='oct_mismatch',hover_name=plot_df.index)
# fig.show()

In [ ]:
plot_df[plot_df["Ehull_avg"] < 0.15]

,element.B1,element.B2,element.X,tau_factor,gen_t_factor,oct_mismatch,GRACE-3L-OMAT-large-ft-AM,SevenNet-MPALOE,SevenNet-omat24,PET-OAM-XL,Ehull_avg,Eform_avg
1000_CsAgLuCl,Ag,Lu,Cl,3.787884,0.926126,0.079834,0.000000,0.000000,0.000000,0.000000,0.000000e+00,-1.918685
1002_CsNaPrCl,Na,Pr,Cl,3.787892,0.926894,0.008287,0.000658,0.003897,0.000000,0.000113,1.166987e-03,-2.150775
1003_CsAgDyCl,Ag,Dy,Cl,3.787892,0.926777,0.035912,0.000000,0.000000,0.000000,0.000000,0.000000e+00,-1.917249
1004_CsMgNdCl,Mg,Nd,Cl,3.787892,0.924534,0.157459,0.127597,0.088584,0.139095,0.129547,1.212058e-01,-1.969830
1007_CsAgYbCl,Ag,Yb,Cl,3.787901,0.925007,0.077901,0.066790,0.000000,0.000000,0.057411,3.105033e-02,-1.800209
...,...,...,...,...,...,...,...,...,...,...,...,...
3748_CsCsIrCl,Cs,Ir,Cl,3.944655,0.868164,0.273481,0.021435,0.010647,0.024037,0.023603,1.993033e-02,-1.443942
3749_CsCsIrBr,Cs,Ir,Br,4.072314,0.860773,0.252551,0.041644,0.033470,0.045753,0.041407,4.056835e-02,-1.245431
3750_CsCsAuF,Cs,Au,F,3.784222,0.870936,0.308271,0.061724,0.073405,0.079039,0.060579,6.868664e-02,-2.092450
3752_CsCsTlF,Cs,Tl,F,3.849975,0.865604,0.295113,0.000003,0.000000,0.000000,0.000000,7.182360e-07,-2.467578


In [ ]:
hdp_df.fillna({"element.B2": "Vac"}, inplace=True)
count_b1 = hdp_df.groupby("element.B1").count()["comp_name_simple"]
count_b2 = hdp_df.groupby("element.B2").count()["comp_name_simple"]

idx_comb = pd.Index(
    set(count_b2.index.to_list() + count_b1.index.to_list())
).sort_values()
count_df = count_b1.reindex(idx_comb).fillna(0) + count_b2.reindex(idx_comb).fillna(0)
fig = px.histogram(count_df, x=count_df.index, y="comp_name_simple")
fig.show()

Gtk-Message: 19:01:07.601: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.
Gtk-Message: 19:01:07.660: Failed to load module "canberra-gtk-module"
Gtk-Message: 19:01:07.661: Failed to load module "canberra-gtk-module"


In [ ]:
def comparative_histograms(
    hdp_df: pd.DataFrame,
    base_indices: list | pd.Index | pd.Series,
    compare_indices: list | pd.Index | pd.Series,
    base_label: str = "full set",
    compare_label: str = "unstable comps",
    **kwargs
):
    import plotly.graph_objects as go
    from pymatgen.core import Element

    col = hdp_df.columns.to_list()[0]
    base_count_b1 = hdp_df.loc[base_indices].groupby("element.B1").count()[col]
    base_count_b2 = hdp_df.loc[base_indices].groupby("element.B2").count()[col]

    elem_list = list(set(base_count_b2.index.to_list() + base_count_b1.index.to_list()))
    elem_map = {elem : Element(elem).Z for elem in elem_list if elem != 'Vac'}
    elem_map.update({'Vac':1})

    elem_idx = pd.Index(
        dict(sorted(elem_map.items() , key= lambda item: item[1])).keys()
    )
    print(elem_idx)

    base_count_ser = base_count_b1.reindex(elem_idx).fillna(0) + base_count_b2.reindex(
        elem_idx
    ).fillna(0)

    compare_count_b1 = hdp_df.loc[compare_indices].groupby("element.B1").count()[col]
    compare_count_b2 = hdp_df.loc[compare_indices].groupby("element.B2").count()[col]
    compare_count_ser = compare_count_b1.reindex(elem_idx).fillna(0) + compare_count_b2.reindex(elem_idx).fillna(0)

    counter_df = pd.DataFrame({base_label: base_count_ser, compare_label: compare_count_ser})
    fig = go.Figure(data = [
        go.Bar(x=elem_idx, y = counter_df[base_label]/np.sum(counter_df[base_label]), name=base_label, marker_color="#575554"),
        go.Bar(x=elem_idx, y = counter_df[compare_label]/np.sum(counter_df[compare_label]), name=compare_label, marker_color="#E55107"),
    ])
    fig.update_traces(opacity=0.75)
    fig.show()

In [76]:
dict(sorted(elem_map.items() , key= lambda item: item[1])).keys()

dict_keys(['Vac', 'Li', 'Be', 'B', 'N', 'Na', 'Mg', 'Al', 'Si', 'P', 'K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As', 'Se', 'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te', 'Cs', 'Ba', 'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu', 'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Po', 'Fr', 'Ra', 'Ac', 'Th', 'Pa', 'U', 'Np', 'Pu', 'Am', 'Cm'])

In [88]:
comparative_histograms(hdp_df=hdp_df, base_indices=hdp_df.index, compare_indices=plot_df[plot_df['Ehull_avg']> 0.2].index)

Index(['Vac', 'Li', 'Be', 'B', 'N', 'Na', 'Mg', 'Al', 'Si', 'P', 'K', 'Ca',
       'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge',
       'As', 'Se', 'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd',
       'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te', 'Cs', 'Ba', 'La', 'Ce', 'Pr', 'Nd',
       'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu', 'Hf',
       'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Po',
       'Fr', 'Ra', 'Ac', 'Th', 'Pa', 'U', 'Np', 'Pu', 'Am', 'Cm'],
      dtype='str')
     full set  unstable comps
Vac     132.0             1.0
Li      109.0             3.0
Be       48.0            35.0
B        22.0            18.0
N        19.0            12.0
Na      154.0             3.0
Mg       68.0             7.0
Al       31.0             1.0
Si        1.0             0.0
P        27.0             8.0


Gtk-Message: 21:34:36.216: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.
Gtk-Message: 21:34:36.276: Failed to load module "canberra-gtk-module"
Gtk-Message: 21:34:36.277: Failed to load module "canberra-gtk-module"


In [ ]:
mlip_df["chemsys"].iloc[-10:]

3745_CsCsTaCl    Cl-Cs-Ta
3746_CsCsTaBr    Br-Cs-Ta
3747_CsCsIrF      Cs-F-Ir
3748_CsCsIrCl    Cl-Cs-Ir
3749_CsCsIrBr    Br-Cs-Ir
3750_CsCsAuF      Au-Cs-F
3751_CsCsAuCl    Au-Cl-Cs
3752_CsCsTlF      Cs-F-Tl
3754_CsCsNpF      Cs-F-Np
3756_CsCsAmF      Am-Cs-F
Name: chemsys, dtype: str

In [ ]:
mlip_df.drop("structure_dict", axis=1).drop("structure", axis=1).to_csv(
    "expensivemlips_energyov_nostruc.csv"
)

In [ ]:
ehull_df.drop("structure_dict", axis=1).to_csv("ExpensiveMLIPs_Ehulldata_noStruc.csv")

KeyError: "['structure_dict'] not found in axis"

In [ ]:
import json

with open("ExpensiveMLIPs/mpid_energy_dict_grace.json") as f:
    d1 = json.load(f)

with open("ExpensiveMLIPs/mpid_energy_dict_GRACE-3L-OMAT-large-ft-AM.json") as f:
    d2 = json.load(f)
d2.update(d1)


with open("ExpensiveMLIPs/mpid_energy_dict_GRACE.json", "w") as f:
    json.dump(d2, f)

In [ ]:
mlip_df.loc["1006_CsPtAmCl"]

structure_dict     {'@module': 'pymatgen.core.structure', '@class...
chemsys                                                  Am-Cl-Cs-Pt
nsites                                                            10
E_MatPES-r2SCAN                                                  NaN
E_MACE-MP-0b3                                                    NaN
E_MACE-MPA-0                                                     NaN
E_SevenNet                                                       NaN
structure          [[0. 0. 0.] Pt, [5.22117225 5.22117225 5.22117...
Name: 1006_CsPtAmCl, dtype: object

In [ ]:
[x.split("_")[-3] for x in glob.glob("MLIP_2retry/*batch_0.json")]

['MatPES-r2SCAN', 'MACE-MP-0b3', 'MACE-MPA-0', 'SevenNet']

In [ ]:
diagram_dir = "MLIP_PhaseDiagrams_20250720"
os.makedirs(diagram_dir, exist_ok=True)

In [ ]:
ehull_df[ehull_df["MACE-Wang"] == 0]

,MACE-Gabor,MACE-Wang
1000_CsAgLuCl,0.0,0.0
1002_CsNaPrCl,0.0,0.0
1003_CsAgDyCl,0.0,0.0
1004_CsMgNdCl,0.0,0.0
1011_CsCaYbCl,0.0,0.0
...,...,...
3739_CsCsHoF,0.0,0.0
3740_CsCsErF,0.0,0.0
3742_CsCsYbF,0.0,0.0
3743_CsCsLuF,0.0,0.0


In [ ]:
ehull_df = get_e_above_hull_df(hdp_df, mlip_df, diagram_output_dir=None)

MACE-MPA-0                           MACE-MP-0b3  \
              E_above_hull_per_atom E_form_per_atom E_above_hull_per_atom   
1000_CsAgLuCl              0.000000       -1.861481              0.000000   
1001_CsAuVCl               0.146274       -1.281205              0.127443   
1002_CsNaPrCl              0.000000       -2.116031              0.001293   
1003_CsAgDyCl              0.000000       -1.860123              0.000000   
1004_CsMgNdCl              0.118294       -1.910491              0.000000   

                                      MatPES-r2SCAN                  \
              E_form_per_atom E_above_hull_per_atom E_form_per_atom   
1000_CsAgLuCl       -1.850720              0.000000       -2.124572   
1001_CsAuVCl        -1.289001              0.113728       -1.577343   
1002_CsNaPrCl       -2.110716              0.000000       -2.392588   
1003_CsAgDyCl       -1.853998              0.000000       -2.064005   
1004_CsMgNdCl       -2.075359              0.000000       -2.417246   

                           SevenNet                  
              E_above_hull_per_atom E_form_per_atom  
1000_CsAgLuCl              0.000000       -1.868575  
1001_CsAuVCl               0.152083       -1.296619  
1002_CsNaPrCl              0.000000       -2.129687  
1003_CsAgDyCl              0.000000       -1.873152  
1004_CsMgNdCl              0.023339       -2.017622

In [ ]:
ehull_df = ehull_df.round(4)
ehull_df.to_csv(f"{diagram_dir}/HDP_EHull_EForm_overview.csv")
d_ehull = ehull_df.xs("E_above_hull_per_atom", axis=1, level=1)
d_ehull.to_csv(f"{diagram_dir}/HDP_EHull_4MLIP.csv")
d_eform = ehull_df.xs("E_form_per_atom", axis=1, level=1)
d_eform.to_csv(f"{diagram_dir}/HDP_EForm_4MLIP.csv")

In [ ]:
d_ehull = ehull_df.round(4)
print("MLIP:", "\t", "NaN vals:", "\t", "share unstable:")

for mlip in d_ehull.columns:
    sclean = d_ehull[mlip].dropna()
    dropped_cols = len(d_ehull) - len(sclean)
    non_stalbe = sclean[sclean >= 0.1]
    print(mlip, "\t", dropped_cols, "\t", len(non_stalbe) / len(sclean))

MLIP: 	 NaN vals: 	 share unstable:
MACE-MPA-0 	 256 	 0.27340332458442695
MACE-MP-0b3 	 257 	 0.21312910284463896
MatPES-r2SCAN 	 258 	 0.16856392294220665
SevenNet 	 256 	 0.23622047244094488


In [ ]:
print("MLIP:", "\t", "NaN vals:")
for mlip in [x for x in mlip_df.columns if x.startswith("E_")]:
    all_ens = mlip_df[mlip].dropna()
    dropped_comps = len(mlip_df) - len(all_ens)
    print(mlip, "\t", dropped_comps)

MLIP: 	 NaN vals:
E_MACE-MPA-0 	 271
E_MACE-MP-0b3 	 276
E_MatPES-r2SCAN 	 389
E_SevenNet 	 270


In [ ]:
d_ehull.dropna(how="all")

,MACE-MPA-0,MACE-MP-0b3,MatPES-r2SCAN,SevenNet
1000_CsAgLuCl,0.0000,0.0000,0.0000,0.0000
1001_CsAuVCl,0.1463,0.1274,0.1137,0.1521
1002_CsNaPrCl,0.0000,0.0013,0.0000,0.0000
1003_CsAgDyCl,0.0000,0.0000,0.0000,0.0000
1004_CsMgNdCl,0.1183,0.0000,0.0000,0.0233
...,...,...,...,...
3749_CsCsIrBr,0.0424,0.0622,0.1100,0.0661
3750_CsCsAuF,0.0559,0.0492,0.0846,0.0574
3751_CsCsAuCl,0.1391,0.1245,0.1117,0.1360
3752_CsCsTlF,0.0000,0.0000,0.0000,0.0000


In [93]:
hdp_df[hdp_df['popdiff.total']>0.1].sample(5).index.to_list()

['2314_CsNaFeF',
 '2160_CsCrCdF',
 '3066_CsYbAmBr',
 '1426_CsNaAgCl',
 '1606_CsLiErCl']

In [9]:
hdp_df[hdp_df['cond_type']=='half-metal'].sample(5).index.to_list()

['2347_CsYbAmF',
 '1198_CsNiSmCl',
 '3551_CsPtAmI',
 '3443_CsFeTlI',
 '2828_CsMnAuBr']